In [1]:
%run shared_imports.py
from decimal import Decimal
from dataclasses import dataclass
from sqlalchemy.orm.query import Query
import gspread
from gspread_dataframe import get_as_dataframe, set_with_dataframe


In [2]:
%run ss13_walker.py

In [3]:
@dataclass(frozen=True)
class SpawnerData:
    feedback_key: str
    feedbacks: list[Feedback]

    def total_spawns(self):
        return sum((spawns for feedback in self.feedbacks for spawns in feedback.values()))

    def spawns_per_round(self):
        return [sum(x.values()) for x in self.feedbacks]

    def counts(self):
        counts = defaultdict(list)
        for feedback in self.feedbacks:
            for name, count in feedback.items():
                counts[name].append(count)
        return counts

    def average(self, path):
        return mean(x for x in self.counts().get(path, [0]))

    def spawn_paths(self):
        return {key for feedback in self.feedbacks for key in feedback.keys() if key}

    def max_round_count(self, path):
        return max(x for x in self.counts().get(path, [0]))

    def mode(self, path):
        return mode(x for x in self.counts().get(path, [0]))

    def means(self):
        means = {name: round(Decimal(sum(count) / len(self.feedbacks)), 2) for name, count in self.counts().items()}
        return sorted(means.items(), key = lambda x: x[1])

    def total_count(self, path):
        return sum(x for x in self.counts().get(path, []))

    def effective_percent(self, path):
        return sum([x / self.total_spawns() for x in self.counts().get(path, [])])

In [4]:
pre_ss13 = SS13("D:/ExternalRepos/third_party/ParadiseMaster/paradise.dme")

In [5]:
td = pre_ss13.dme.typedecl('/obj/effect/spawner/lootdrop/maintenance')

In [6]:
old_testmap =  DMM.from_file("D:/ExternalRepos/third_party/ParadiseMaster/_maps/map_files/stations/metastation.dmm")

In [7]:
old_boxstation = DMM.from_file("D:/ExternalRepos/third_party/ParadiseMaster/_maps/map_files/stations/boxstation.dmm")

In [8]:
old_spawners = defaultdict(int)
for coord in old_testmap.coords():
    tile = old_testmap.tiledef(*coord)
    for spawner in tile.find("/obj/effect/spawner/lootdrop/maintenance"):
        old_spawners[tile.prefab_path(spawner)] += 1

In [229]:
ROUND_COUNT = 50

In [479]:
old_spawners

defaultdict(int,
            {/obj/effect/spawner/lootdrop/maintenance: 187,
             /obj/effect/spawner/lootdrop/maintenance/three: 15,
             /obj/effect/spawner/lootdrop/maintenance/two: 25,
             /obj/effect/spawner/lootdrop/maintenance/eight: 2})

In [480]:
runs_old = list()
lootcounts = dict()
for spawner in old_spawners.keys():
    lootcounts[spawner] = pre_ss13.dme.typedecl(spawner).value('lootcount')
loot_table = pre_ss13.dme.typedecl('/obj/effect/spawner/lootdrop/maintenance').value('loot')
for i in range(ROUND_COUNT):
    run = defaultdict(int)
    for spawner, count in old_spawners.items():
        for j in range(count):
            for k in range(lootcounts[spawner]):
                typepath = pick_weight_recursive(loot_table)
                if not typepath:
                    continue
                if isinstance(typepath, Prefab):
                    typepath = typepath.path

                if str(typepath).startswith('/obj/item/stack/rods'):
                    typepath = p('/obj/item/stack/rods')
                elif str(typepath).startswith('/obj/item/stack/cable_coil'):
                    typepath = p('/obj/item/stack/cable_coil')
                elif str(typepath).startswith('/obj/item/stack/sheet/cloth'):
                    typepath = p('/obj/item/stack/sheet/cloth')
                elif str(typepath).startswith('/obj/item/stack/sheet/metal'):
                    typepath = p('/obj/item/stack/sheet/metal')
                    
                run[typepath] += 1
    runs_old.append(run)

pre_spawn = SpawnerData('random_spawners_pre', runs_old)

In [43]:
pre_spawn.total_count('/obj/item/extinguisher')

231

In [44]:
print("{:.2%}".format(pre_spawn.effective_percent('/obj/item/extinguisher')))

5.30%


In [473]:
new_testmap = DMM.from_file("D:/ExternalRepos/third_party/Paradise/_maps/map_files/stations/boxstation.dmm")

In [9]:
new_boxstation = DMM.from_file("D:/ExternalRepos/third_party/Paradise/_maps/map_files/stations/boxstation.dmm")

In [10]:
new_spawners = defaultdict(int)
for coord in new_boxstation.coords():
    tile = new_boxstation.tiledef(*coord)
    for spawner in tile.find("/obj/effect/spawner/random/maintenance"):
        new_spawners[tile.prefab_path(spawner)] += 1

In [11]:
new_spawners

defaultdict(int, {/obj/effect/spawner/random/maintenance: 298})

In [28]:
mean(sum([random.randint(2, 4) for _ in range(298) if prob(65)]) for _ in range(9999)) * 0.75

435.744599459946

In [16]:
sum(runs_new[0].values())

NameError: name 'runs_new' is not defined

In [373]:
mean(post_spawn.spawns_per_round())

580.92

In [374]:
mean(pre_spawn.spawns_per_round())

418.88

In [101]:
mean(post_spawn.spawns_per_round())

425.9

In [410]:
total_count = 0
for spawner, count in new_spawners.items():
    for j in range(count):
        if prob(30):
            total_count += random.randint(2, 4)
            

In [411]:
total_count

283

In [9]:
pre_spawn.effective_percent('/obj/item/extinguisher')

0.055383884002852386

In [10]:
post_spawn.effective_percent('/obj/item/extinguisher')

0.006701414743112435

In [158]:
gc = gspread.service_account()
sh = gc.open("Maint Loot Relootening")

In [415]:
worksheet = sh.get_worksheet(2)

In [417]:
worksheet = sh.add_worksheet(title="Filled Maints: Trash Reduction", rows=len(all_paths), cols=7)

In [397]:
worksheet = sh.add_worksheet(title="Empty Maints", rows=len(all_paths), cols=7)

In [421]:
set_with_dataframe(worksheet, df)

In [423]:
from statistics import mode

In [494]:
ss13 = SS13("D:/ExternalRepos/third_party/Paradise/paradise.dme")
maintenance_loot_tables = ss13.global_list('maintenance_loot_tables')

runs_new = list()
for i in range(ROUND_COUNT):
    run = defaultdict(int)
    for spawner, count in new_spawners.items():
        for j in range(count):
            spawn_loot_chance = ss13.dme.typedecl(spawner).value('spawn_loot_chance')
            if prob(spawn_loot_chance):
                loot_spawned = 0
                spawn_loot_count = random.randint(2, 4)
                while (spawn_loot_count - loot_spawned):
                    typepath = ss13.super_recursive_pick(maintenance_loot_tables)
                    tp = typepath.path if isinstance(typepath, Prefab) else typepath
                    if tp.child_of('/obj/item/book'):
                        tp = p('/obj/item/book')
                    if tp.child_of('/obj/item/trash') or tp.child_of('/obj/item/shard'):
                        tp = p('/obj/item/trash')
                    run[tp] += 1
                    loot_spawned += 1
    runs_new.append(run)

post_spawn = SpawnerData('random_spawners_post', runs_new)

In [495]:
all_paths = pre_spawn.spawn_paths().union(post_spawn.spawn_paths())
shared_paths = post_spawn.spawn_paths().intersection(pre_spawn.spawn_paths())

rows = []

for path in sorted(all_paths):
    rows.append([
        path,
        round(pre_spawn.average(path), 2),
        round(post_spawn.average(path), 2),
        pre_spawn.mode(path),
        post_spawn.mode(path),
        pre_spawn.max_round_count(path),
        post_spawn.max_round_count(path),
        round(Decimal(pre_spawn.effective_percent(path) * 100.0), 2),
        round(Decimal(r.effective_percent(str(path)) * 100), 2),
    ])

df = pd.DataFrame(rows, columns=[
    "Type Path",
    f"Before: Average, {len(pre_spawn.feedbacks)} rounds",
    f"After: Average, {len(post_spawn.feedbacks)} rounds",    
    f"Before: Mode",
    f"After: Mode",
    f"Before: Max",
    f"After: Max",
    "Before: Effective Percent",
    "After: Effective Percent",
])

In [496]:
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(df.sort_values('Type Path'))

,Type Path,"Before: Average, 50 rounds","After: Average, 50 rounds",Before: Mode,After: Mode,Before: Max,After: Max,Before: Effective Percent,After: Effective Percent
0,/obj/effect/spawner/random_spawners/mod/maint,2.38,0.00,1,0,6,0,0.65,0.00
1,/obj/item/ammo_box/magazine/m10mm,1.24,1.60,1,1,3,4,0.22,0.00
2,/obj/item/ammo_casing/c10mm,0.00,2.96,0,3,0,10,0.00,0.00
3,/obj/item/analyzer,0.00,4.52,0,5,0,12,0.00,2.24
4,/obj/item/assembly/prox_sensor,8.02,1.71,7,1,15,6,2.79,0.60
5,/obj/item/assembly/signaler,0.00,1.86,0,1,0,6,0.00,0.30
6,/obj/item/assembly/timer,5.54,2.00,5,2,10,4,1.93,0.30
7,/obj/item/assembly/voice,0.00,1.71,0,1,0,4,0.00,0.00
8,/obj/item/assembly/voice/noise,0.00,1.72,0,1,0,4,0.00,0.15
9,/obj/item/bio_chip_implanter/storage,1.00,1.00,1,1,1,1,0.08,0.00


In [120]:
len(post_spawn.feedbacks)

6

In [49]:
feedbacks = session.query(Feedback).join(Round).filter(Feedback.key_name=='random_spawners', Round.id >= 1938)

In [50]:
counts = defaultdict(list)
for feedback in feedbacks.all():
    for name, count in feedback.items():
        counts[name].append(count)

In [58]:
means = {name: round(decimal.Decimal(sum(count) / feedbacks.count()), 2) for name, count in counts.items()}

In [60]:
feedbacks.count()

74

In [61]:
for name, amt in :
    print(f"{name}: {amt}")

/obj/item/deck/cards/syndicate: 0.18
/obj/item/grenade/smokebomb: 0.26
/obj/item/gun/syringe/syndicate: 0.26
/obj/item/stamp/chameleon: 0.28
/obj/item/storage/secure/briefcase/syndie: 0.28
/obj/item/storage/backpack/satchel_flat: 0.30
/obj/item/clothing/suit/jacket/bomber/syndicate: 0.30
/obj/item/clothing/mask/chameleon: 0.32
/obj/item/grenade/clown_grenade: 0.32
/obj/item/storage/belt/military/traitor: 0.32
/obj/item/weaponcrafting/receiver: 0.32
/obj/item/storage/pill_bottle/fakedeath: 0.34
/obj/item/clothing/suit/storage/iaa/blackjacket/armored: 0.34
/obj/item/clothing/under/chameleon: 0.35
/obj/item/clothing/mask/gas/voice_modulator: 0.35
/obj/item/storage/backpack/duffel/syndie/med/surgery_fake: 0.38
/obj/item/suppressor: 0.38
/obj/item/ammo_box/magazine/m10mm: 0.39
/obj/item/clothing/mask/gas/voice_modulator/chameleon: 0.39
/obj/item/dnascrambler: 0.42
/obj/item/mod/construction/broken_core: 0.42
/obj/item/storage/toolbox/syndicate: 0.45
/obj/item/clothing/mask/chameleon/voice_c

In [71]:
engine = make_engine("settings.toml")
session = Session(engine)


In [45]:
session.query(Feedback).join(Round).where(
    and_(
        Feedback.key_name == 'testmerged_prs',
        Feedback.json["data"].regexp_match("25619"),
        Round.start_datetime >= datetime(2024,8,6)
    )
).count()


73

In [87]:
sum(session.query(Feedback).join(Round).filter(Feedback.key_name=='random_spawners', Round.id == 2027)[0].values())


555

In [76]:
sum(session.query(Feedback).join(Round).filter(Feedback.key_name=='random_spawners', Round.id == 2019)[0].values())


312

In [90]:
sum(session.query(Feedback).join(Round).filter(Feedback.key_name=='random_spawners', Round.id == 2029)[0].values())


142

In [106]:
with Session(engine) as session:
    feedbacks = session.query(Feedback).join(Round).filter(Feedback.key_name=='random_spawners', Round.id >= 2033)
    total_spawns = sum((spawns for feedback in feedbacks.all() for spawns in feedback.values()))
    counts = defaultdict(list)
    for feedback in feedbacks.all():
        for name, count in feedback.items():
            counts[name].append(count)
    means = {name: round(decimal.Decimal(sum(count) / feedbacks.count()), 2) for name, count in counts.items()}
    for name, amt in sorted(means.items(), key = lambda x: x[1]):
        print(f"{name}: {amt}")

/obj/item/storage/fancy/cigarettes/cigpack_syndicate: 0.50
/obj/item/grenade/smokebomb: 0.50
/obj/item/reagent_containers/dropper: 0.50
/obj/item/storage/backpack/satchel_flat: 0.50
/obj/item/seeds/ambrosia/cruciatus: 0.50
/obj/item/radio/off: 0.50
/obj/item/deck/cards/syndicate: 0.50
/obj/item/clothing/gloves/color/fyellow: 0.50
/obj/item/tank/internals/emergency_oxygen: 0.50
/obj/item/assembly/signaler: 0.50
/obj/item/reagent_containers/glass/bucket: 0.50
/obj/item/clothing/suit/storage/iaa/blackjacket/armored: 0.50
/obj/item/stamp/chameleon: 0.50
/obj/item/stock_parts/capacitor: 0.50
/obj/item/storage/pill_bottle/fakedeath: 0.50
/obj/item/storage/wallet: 0.50
/obj/item/stock_parts/micro_laser: 0.50
/obj/item/gun/projectile/automatic/pistol: 0.50
/obj/item/assembly/prox_sensor: 0.50
/obj/item/tank/internals/emergency_oxygen/engi: 0.50
/obj/item/storage/firstaid/regular: 0.50
/obj/item/reagent_scanner/adv: 0.50
/obj/item/melee/knuckleduster: 0.50
/obj/item/robotanalyzer: 0.50
/obj/ite

In [278]:
test = json.loads("""
{"/obj/item/clothing/glasses/meson":11,"/obj/item/whetstone":2,"/obj/item/stack/sheet/plastic":5,"/obj/item/mod/construction/broken_core":3,"/obj/item/clothing/head/hardhat":1,"/obj/item/crowbar/red":17,"/obj/item/assembly/prox_sensor":4,"/obj/item/storage/firstaid/machine":1,"/obj/item/reagent_containers/glass/bucket":1,"/obj/item/analyzer":8,"/obj/item/seeds/ambrosia/deus":2,"/obj/item/stack/sheet/wood":14,"/obj/item/stack/sheet/metal":16,"/obj/item/grenade/smokebomb":2,"/obj/item/stack/cable_coil":12,"/obj/item/storage/fancy/crayons":6,"/obj/item/stock_parts/matter_bin":3,"/obj/item/mod/module/stamp":4,"/obj/item/clothing/mask/gas/voice_modulator":3,"/obj/item/food/pistachios":1,"/obj/item/clothing/suit/storage/hazardvest":1,"/obj/item/scratch":4,"/obj/item/storage/toolbox/mechanical":1,"/obj/item/storage/box":1,"/obj/item/wrench":9,"/obj/item/crowbar":10,"/obj/item/storage/wallet":7,"/obj/item/reagent_containers/drinks/mug":3,"/obj/item/stack/sheet/mineral/plasma":3,"/obj/item/restraints/handcuffs/toy":5,"/obj/item/camera_film":2,"/obj/item/stack/sheet/rglass":17,"/obj/item/weldingtool":12,"/obj/item/geiger_counter":9,"/obj/item/storage/wallet/random":5,"/obj/item/trash":60,"/obj/item/storage/backpack/satchel_flat":1,"/obj/item/stack/rods":12,"/obj/item/seeds/ambrosia":4,"/obj/item/storage/belt/utility":12,"/obj/item/wirecutters":7,"/obj/item/clothing/mask/gas":3,"/obj/item/t_scanner":10,"/obj/item/weaponcrafting/receiver":2,"/obj/item/reagent_containers/syringe":2,"/obj/item/clothing/head/that":2,"/obj/item/storage/box/lights/mixed":3,"/obj/item/assembly/signaler":2,"/obj/item/clothing/under/chameleon":4,"/obj/item/caution":4,"/obj/item/tank/internals/emergency_oxygen":3,"/obj/item/stack/sheet/glass":15,"/obj/item/stamp/chameleon":2,"/obj/item/cultivator":4,"/obj/item/clothing/suit/jacket/bomber/syndicate":4,"/obj/item/screwdriver":9,"/obj/item/stack/sheet/plasteel":9,"/obj/item/clothing/gloves/color/yellow/fake":3,"/obj/item/mod/module/balloon":4,"/obj/item/ammo_box/magazine/m10mm":4,"/obj/item/clothing/head/cone":6,"/obj/item/storage/fancy/matches":2,"/obj/item/flashlight":5,"/obj/item/bodybag":3,"/obj/item/storage/box/donkpockets":2,"/obj/item/clothing/head/welding":11,"/obj/item/storage/pill_bottle/fakedeath":4,"/obj/item/clothing/under/misc/vice":3,"/obj/item/vending_refill/cola":4,"/obj/item/stack/medical/bruise_pack/advanced":3,"/obj/item/clothing/gloves/color/black":5,"/obj/item/poster/random_official":2,"/obj/item/stack/sheet/cardboard":7,"/obj/item/clothing/under/color/black":7,"/obj/item/mod/module/springlock":3,"/obj/item/multitool/ai_detect":1,"/obj/item/light/tube":1,"/obj/item/melee/knuckleduster/syndie":2,"/obj/item/stock_parts/cell":2,"/obj/item/storage/secure/briefcase/syndie":3,"/obj/item/gun/projectile/automatic/pistol":1,"/obj/item/clothing/head/ushanka":2,"/obj/item/assembly/voice/noise":2,"/obj/item/radio/off":2,"/obj/item/gun/syringe/syndicate":1,"/obj/item/clothing/head/hardhat/red":3,"/obj/item/seeds/ambrosia/cruciatus":2,"/obj/item/food/stroopwafel":2,"/obj/item/storage/firstaid/regular":3,"/obj/item/soap/syndie":3,"/obj/item/storage/belt/military/traitor":3,"/obj/item/clothing/gloves/color/yellow":5,"/obj/item/stock_parts/manipulator":1,"/obj/item/reagent_containers/dropper":3,"/obj/item/clothing/shoes/chameleon/noslip":2,"/obj/item/stack/sheet/cloth":2,"/obj/item/light/bulb":2,"/obj/item/food/cheesiehonkers":1,"/obj/item/camera":4,"/obj/item/storage/backpack/duffel/syndie/med/surgery_fake":1,"/obj/item/grenade/clown_grenade":1,"/obj/item/assembly/voice":1,"/obj/item/clothing/gloves/color/fyellow":1,"/obj/item/clothing/suit/storage/iaa/blackjacket/armored":2,"/obj/item/multitool":5,"/obj/item/dice/d10":1,"/obj/item/dice/d8":1,"/obj/item/storage/toolbox/emergency":1,"/obj/item/food/tastybread":3,"/obj/item/tank/internals/emergency_oxygen/engi":1,"/obj/item/clothing/shoes/black":2,"/obj/item/reagent_containers/glass/beaker/waterbottle":1,"/obj/item/assembly/timer":1,"/obj/item/dice/d12":1,"/obj/item/scissors":1,"/obj/item/pen/red":1,"/obj/item/book":2,"/obj/item/clothing/glasses/sunglasses":2,"/obj/item/storage/box/cups":1,"/obj/item/clothing/mask/gas/voice_modulator/chameleon":1,"/obj/item/storage/fancy/cigarettes/dromedaryco":1,"/obj/item/relic":2,"/obj/item/flashlight/pen":2,"/obj/item/reagent_containers/spray/pestspray":1,"/obj/item/reagent_containers/glass/beaker/large":3,"/obj/item/storage/bag/plasticbag":2,"/obj/item/storage/toolbox/syndicate":2,"/obj/item/robotanalyzer":1,"/obj/item/suppressor":2,"/obj/item/dice/d4":1}
""")

In [308]:
r = SpawnerData('r', [])

In [309]:
r.feedbacks.append(json.loads("""
{"/obj/item/trash":258,"/obj/item/flashlight":4,"/obj/item/stack/sheet/rglass":16,"/obj/item/screwdriver":16,"/obj/item/stack/sheet/metal":9,"/obj/item/clothing/head/welding":18,"/obj/item/crowbar":9,"/obj/item/stack/rods":14,"/obj/item/multitool":4,"/obj/item/stack/medical/ointment/advanced":3,"/obj/item/clothing/suit/storage/iaa/blackjacket/armored":3,"/obj/item/t_scanner":4,"/obj/item/multitool/ai_detect":1,"/obj/item/stack/sheet/cloth":1,"/obj/item/stack/sheet/mineral/plasma":5,"/obj/item/storage/wallet/random":4,"/obj/item/storage/pill_bottle/fakedeath":4,"/obj/item/wirecutters":9,"/obj/item/storage/backpack/satchel_flat":3,"/obj/item/crowbar/red":9,"/obj/item/weldingtool":8,"/obj/item/clothing/mask/chameleon":1,"/obj/item/stack/cable_coil":11,"/obj/item/cultivator":3,"/obj/item/grenade/smokebomb":2,"/obj/item/analyzer":15,"/obj/item/tank/internals/emergency_oxygen":5,"/obj/item/stack/sheet/plastic":6,"/obj/item/weaponcrafting/receiver":1,"/obj/item/stack/sheet/glass":11,"/obj/item/storage/belt/military/traitor":4,"/obj/item/clothing/head/ushanka":2,"/obj/item/storage/toolbox/electrical":1,"/obj/item/assembly/timer":2,"/obj/item/coin/twoheaded":2,"/obj/item/storage/box/donkpockets":3,"/obj/item/stack/sheet/plasteel":8,"/obj/item/stack/sheet/wood":13,"/obj/item/storage/belt/utility":6,"/obj/item/clothing/under/color/black":4,"/obj/item/reagent_containers/glass/beaker/large":1,"/obj/item/mod/module/stamp":3,"/obj/item/clothing/glasses/meson":9,"/obj/item/stack/medical/bruise_pack/advanced":3,"/obj/item/book":1,"/obj/item/clothing/under/misc/vice":3,"/obj/item/suppressor":2,"/obj/item/melee/knuckleduster":2,"/obj/item/clothing/mask/chameleon/voice_change":1,"/obj/item/storage/fancy/crayons":2,"/obj/item/storage/box/lights/mixed":2,"/obj/item/vending_refill/cola":2,"/obj/item/geiger_counter":9,"/obj/item/assembly/prox_sensor":4,"/obj/item/light/tube":3,"/obj/item/reagent_containers/drinks/mug":3,"/obj/item/soap/syndie":2,"/obj/item/clothing/shoes/chameleon/noslip":3,"/obj/item/reagent_containers/syringe":2,"/obj/item/clothing/gloves/color/yellow":1,"/obj/item/stock_parts/capacitor":1,"/obj/item/storage/fancy/cigarettes/cigpack_syndicate":2,"/obj/item/hand_labeler":2,"/obj/item/restraints/handcuffs/toy":4,"/obj/item/clothing/gloves/color/yellow/fake":2,"/obj/item/wrench":8,"/obj/item/storage/box/cups":2,"/obj/item/melee/knuckleduster/syndie":2,"/obj/item/storage/firstaid/machine":2,"/obj/item/reagent_containers/dropper":1,"/obj/item/storage/wallet":7,"/obj/item/storage/toolbox/syndicate":1,"/obj/item/scissors":3,"/obj/item/seeds/ambrosia":2,"/obj/item/stock_parts/cell":2,"/obj/item/flashlight/pen":1,"/obj/item/mod/module/springlock":7,"/obj/item/grenade/clown_grenade":2,"/obj/item/radio/headset":1,"/obj/item/clothing/head/cone":1,"/obj/item/clothing/mask/gas/voice_modulator":1,"/obj/item/assembly/signaler":2,"/obj/item/reagent_containers/glass/bucket":4,"/obj/item/seeds/ambrosia/deus":2,"/obj/item/food/twimsts":1,"/obj/item/food/candy/candybar":1,"/obj/item/storage/box":1,"/obj/item/clothing/gloves/color/fyellow":2,"/obj/item/extinguisher":1,"/obj/item/clothing/head/hardhat":1,"/obj/item/assembly/voice/noise":1,"/obj/item/storage/secure/briefcase/syndie":1,"/obj/item/poster/random_contraband":2,"/obj/item/clothing/under/chameleon":1,"/obj/item/caution":1,"/obj/item/bodybag":4,"/obj/item/seeds/ambrosia/cruciatus":1,"/obj/item/clothing/glasses/sunglasses":1,"/obj/item/gun/syringe/syndicate":1,"/obj/item/tank/internals/emergency_oxygen/engi":1,"/obj/item/stack/nanopaste":2,"/obj/item/storage/backpack/duffel/syndie/med/surgery_fake":1,"/obj/item/stamp/chameleon":2,"/obj/item/mod/module/balloon":1,"/obj/item/clothing/head/hardhat/red":1,"/obj/item/poster/random_official":3,"/obj/item/scratch":1,"/obj/item/light/bulb":1,"/obj/item/storage/toolbox/mechanical":1,"/obj/item/coin/silver":1,"/obj/item/reagent_containers/spray/pestspray":2,"/obj/item/clothing/gloves/color/black":1,"/obj/item/whetstone":1,"/obj/item/storage/firstaid/regular":1,"/obj/item/relic":1,"/obj/item/deck/cards/syndicate":4,"/obj/item/reagent_containers/glass/beaker":1,"/obj/item/food/stroopwafel":1,"/obj/item/food/sosjerky":1,"/obj/item/stack/sheet/cardboard":1}
"""))

In [310]:
mean(r.spawns_per_round())

671

In [311]:
r.max_round_count("/obj/item/trash")

258